# Download the Forsythe GPR lines
The field profiles are the U.S. Geological Survey data release

Fiore, A. R. (2018). *Raw ground-penetrating radar data, Edwin B. Forsythe National Wildlife Refuge, New Jersey, 2014–15.* https://doi.org/10.5066/F7JH3KBD

The ScienceBase item is https://www.sciencebase.gov/catalog/item/5a0c59bce4b09af898cd15ce. It holds ten MALA lines. Each line is a `.rd3` of samples, a `.rad` header, and an `.asc` text export of the same samples. The item also has `markers.txt`, `TraceCoordinates.txt`, and the metadata XML.

This notebook downloads the two files that are used (`DinnerPointAve.rd3` and `DinnerPointAve.rad`) and deletes the unused profiles from `data/raw/usgs_forsythe/`.

The field notes and the metadata XML stay. They describe the survey. `DinnerPointAve_gprpy_steps.npz` is written later so it is not part of this download.


## Setup

`RELEASE` below is the file list published on ScienceBase: name, size in bytes, and download URL. The next cell replaces it when the catalog page answers. ScienceBase often puts a Cloudflare check in front of that page, and a notebook request is then refused. The saved list is what the item contained when this notebook was written, so the later steps still know which names to keep.


In [ ]:
import json
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

def repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "gprpy_reference.py").is_file():
            return candidate
    raise FileNotFoundError("open this notebook from the GPRadar repository")

ROOT = repo_root()
DEST = ROOT / "data" / "raw" / "usgs_forsythe"
CATALOG = "https://www.sciencebase.gov/catalog/item/5a0c59bce4b09af898cd15ce?format=json"
ITEM = "https://www.sciencebase.gov/catalog/item/5a0c59bce4b09af898cd15ce"
# The reference script opens the .rd3. GPRPy looks beside it for the .rad header.
USED = {"DinnerPointAve.rd3", "DinnerPointAve.rad"}
# Survey notes and the release record. They are not a second copy of the samples.
NOTES = {
    "markers.txt",
    "TraceCoordinates.txt",
    "Raw ground-penetrating radar data, Edwin B. Forsythe National Wildlife Refuge, New Jersey, 2014-15.xml",
}

RELEASE = [
    {
        "name": "DinnerPointAve.asc",
        "bytes": 12389895,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__17%2F6f%2F61%2F176f61e91e274ea624b3f17948b102e7de393b8c"
    },
    {
        "name": "DinnerPointAve.rad",
        "bytes": 776,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__c9%2Ff1%2F2b%2Fc9f12ba52d4f5e227dc8c86b5bd4dd90e4629a12"
    },
    {
        "name": "DinnerPointAve.rd3",
        "bytes": 8518392,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__1a%2F05%2F89%2F1a05891cc21360a53e1b19ee11b7a13024b13399"
    },
    {
        "name": "GameFarmRd-1.asc",
        "bytes": 6707849,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__65%2F25%2F84%2F652584794a6d8d580351a8a89e8e556e2c19b72d"
    },
    {
        "name": "GameFarmRd-1.rad",
        "bytes": 789,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__66%2Fb4%2F29%2F66b4299b472adfd546eb74a08718a438732ae84c"
    },
    {
        "name": "GameFarmRd-1.rd3",
        "bytes": 4682268,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__c1%2F9e%2F90%2Fc19e908579a224c16f56332707a6527484927aec"
    },
    {
        "name": "GameFarmRd-2.asc",
        "bytes": 5999765,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__91%2F45%2Fbe%2F9145be8d565752a5e59d3adae6db9bbbd24b41ba"
    },
    {
        "name": "GameFarmRd-2.rad",
        "bytes": 776,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__71%2F1a%2F58%2F711a581f7ab9a23ae343fa6c886c63f3d5d33156"
    },
    {
        "name": "GameFarmRd-2.rd3",
        "bytes": 3939858,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__71%2F69%2Ff5%2F7169f5efae16db80ba24ed05160b00fa26b1a4cb"
    },
    {
        "name": "markers.txt",
        "bytes": 5113,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__46%2Ff0%2Fe3%2F46f0e3601a4819ada5d32c4289f7b10a00da8599"
    },
    {
        "name": "Raw ground-penetrating radar data, Edwin B. Forsythe National Wildlife Refuge, New Jersey, 2014-15.xml",
        "bytes": 21777,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__33%2Fd7%2F37%2F33d7378fa6a2b39dfa4c016feb49a7f1cb6f3f6d"
    },
    {
        "name": "ReedyCreek-1.asc",
        "bytes": 19136851,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__46%2F02%2F18%2F4602182c49550256679f94260149e3f6aeeec5f6"
    },
    {
        "name": "ReedyCreek-1.rad",
        "bytes": 768,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__8f%2F20%2Fe4%2F8f20e448495b968f100e1cde098d5d85d1393c8c"
    },
    {
        "name": "ReedyCreek-1.rd3",
        "bytes": 7555072,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__25%2F82%2Fc2%2F2582c28555fbc389d4928788fa6e236a71d3df24"
    },
    {
        "name": "ReedyCreek-2.asc",
        "bytes": 13760105,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__da%2Fe0%2F7e%2Fdae07e137eb32a1dfd0bcfb54cb2422a2ef60e04"
    },
    {
        "name": "ReedyCreek-2.rad",
        "bytes": 767,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__c7%2Fbc%2F6a%2Fc7bc6a3b6ee2a32f01f9712c4a1d1ecc54b68679"
    },
    {
        "name": "ReedyCreek-2.rd3",
        "bytes": 5433344,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__70%2F41%2F7e%2F70417e4043089fcb6329ae71f18a22f98eb54ba9"
    },
    {
        "name": "ScottsLandingRd.asc",
        "bytes": 12381127,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__8c%2F2b%2F32%2F8c2b3232e7a2fda9166fd93d375a5f72d16a725b"
    },
    {
        "name": "ScottsLandingRd.rad",
        "bytes": 767,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__86%2Fe8%2F89%2F86e889e159f609c62d4275a8969119646e1b6f80"
    },
    {
        "name": "ScottsLandingRd.rd3",
        "bytes": 4890624,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__84%2Fd7%2Fed%2F84d7ed91817cb4d7bf65321e80732e72aa3a8f8b"
    },
    {
        "name": "SouthWildlifeDr.asc",
        "bytes": 6710331,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__84%2F5b%2Ff8%2F845bf8a29713c786f043586ca8bc3dfa36d25fe7"
    },
    {
        "name": "SouthWildlifeDr.rad",
        "bytes": 767,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__b7%2F38%2F85%2Fb73885d83f8b60501a4b384c5f38720bc9ddd58a"
    },
    {
        "name": "SouthWildlifeDr.rd3",
        "bytes": 2648064,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__e6%2Fdb%2F06%2Fe6db06520709aee7999bc4b3dd40e12663f4c17e"
    },
    {
        "name": "StaffordAve-eastwest.asc",
        "bytes": 13954551,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__9a%2F91%2F88%2F9a91883a071203d2e6bbbe2f7b80c335d969034d"
    },
    {
        "name": "StaffordAve-eastwest.rad",
        "bytes": 767,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__b0%2F2e%2Fd3%2Fb02ed3974ce46231276969a053037e11eb8ec752"
    },
    {
        "name": "StaffordAve-eastwest.rd3",
        "bytes": 5505024,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__d7%2Fcf%2F16%2Fd7cf167cb7abdd95cf4c2ec701c94e7eb0533ef4"
    },
    {
        "name": "StaffordAve-westeast.asc",
        "bytes": 13585292,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__ac%2F0e%2Fd3%2Fac0ed349a4239374ccc040afe05692e9c3793569"
    },
    {
        "name": "StaffordAve-westeast.rad",
        "bytes": 767,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__25%2F14%2F6a%2F25146ad17e7019e62ea90d0b9849b7b71b6db0d4"
    },
    {
        "name": "StaffordAve-westeast.rd3",
        "bytes": 5355520,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__26%2Fb2%2Ff1%2F26b2f16d7598478788d13deb79c157fd0283ec4b"
    },
    {
        "name": "TraceCoordinates.txt",
        "bytes": 98034,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__a6%2Faa%2Fd1%2Fa6aad11d377242c990ac8cfabd1a3c147289ba9f"
    },
    {
        "name": "WescottAve.asc",
        "bytes": 8157693,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__10%2F02%2F4d%2F10024d1bc37c1e403662cf614a443c6b3a8ef1e1"
    },
    {
        "name": "WescottAve.rad",
        "bytes": 767,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__98%2Fbe%2F64%2F98be644900c2b3088d4a37b81db69da3c790731b"
    },
    {
        "name": "WescottAve.rd3",
        "bytes": 3221504,
        "url": "https://www.sciencebase.gov/catalog/file/get/5a0c59bce4b09af898cd15ce?f=__disk__6f%2F18%2F4a%2F6f184a330d6f8763d05b3101e1ab6e3b99a9ebdf"
    }
]

DEST.mkdir(parents=True, exist_ok=True)


## 1. Read the file list

A successful catalog request replaces `RELEASE`. A 403 leaves the saved list in place. Nothing is written to disk in this step. `keep` is a file the code reads, `notes` is a survey file that stays, and `delete` is an unused profile or an ASCII copy.


In [ ]:
def fetch_catalog():
    request = Request(CATALOG, headers={"User-Agent": "GPRadar"})
    with urlopen(request, timeout=30) as response:
        catalog = json.load(response)
    rows = []
    for entry in catalog["files"]:
        rows.append({
            "name": entry["name"],
            "bytes": int(entry["size"]),
            "url": entry["downloadUri"],
        })
    return rows

try:
    RELEASE = fetch_catalog()
    print("read the live ScienceBase catalog")
except (HTTPError, URLError, TimeoutError) as error:
    print(f"ScienceBase did not return the catalog ({error}).")
    print("Using the file list saved in this notebook.")
    print(f"The item page is {ITEM}")

release = []
for row in RELEASE:
    name = row["name"]
    if name in USED:
        action = "keep"
    elif name in NOTES:
        action = "notes"
    else:
        action = "delete"
    release.append({**row, "action": action})
release = sorted(release, key=lambda row: row["name"].lower())

print(f"{len(release)} files, {sum(row['bytes'] for row in release) / 1e6:.0f} MB")
print(f"{'action':<8} {'MB':>8}  name")
for row in release:
    print(f"{row['action']:<8} {row['bytes'] / 1e6:8.2f}  {row['name']}")


ScienceBase did not return the catalog (HTTP Error 403: Forbidden).
Using the file list saved in this notebook.
The item page is https://www.sciencebase.gov/catalog/item/5a0c59bce4b09af898cd15ce
33 files, 165 MB
action         MB  name
delete      12.39  DinnerPointAve.asc
keep         0.00  DinnerPointAve.rad
keep         8.52  DinnerPointAve.rd3
delete       6.71  GameFarmRd-1.asc
delete       0.00  GameFarmRd-1.rad
delete       4.68  GameFarmRd-1.rd3
delete       6.00  GameFarmRd-2.asc
delete       0.00  GameFarmRd-2.rad
delete       3.94  GameFarmRd-2.rd3
notes        0.01  markers.txt
notes        0.02  Raw ground-penetrating radar data, Edwin B. Forsythe National Wildlife Refuge, New Jersey, 2014-15.xml
delete      19.14  ReedyCreek-1.asc
delete       0.00  ReedyCreek-1.rad
delete       7.56  ReedyCreek-1.rd3
delete      13.76  ReedyCreek-2.asc
delete       0.00  ReedyCreek-2.rad
delete       5.43  ReedyCreek-2.rd3
delete      12.38  ScottsLandingRd.asc
delete       0.00  ScottsL

## 2. Download the two files that are used

`DinnerPointAve.rd3` is about 8.5 MB. `DinnerPointAve.rad` is the text header. A file already on disk with the published size is not downloaded again.

If ScienceBase refuses the request and the file is missing, save it from the item page in a browser and put it in `data/raw/usgs_forsythe/`, then run this cell again.


In [ ]:
def download(url, path, nbytes):
    if path.is_file() and path.stat().st_size == nbytes:
        print(f"already present  {path.name}  ({nbytes} bytes)")
        return
    request = Request(url, headers={"User-Agent": "GPRadar"})
    try:
        with urlopen(request, timeout=120) as response:
            path.write_bytes(response.read())
    except (HTTPError, URLError, TimeoutError) as error:
        if path.is_file():
            print(f"left the copy of {path.name} already in the folder ({error})")
            return
        raise RuntimeError(
            f"Could not download {path.name}. Save it from {ITEM} into {path.parent}."
        ) from error
    print(f"downloaded  {path.name}  ({path.stat().st_size} bytes)")

by_name = {row["name"]: row for row in release}
for name in sorted(USED):
    row = by_name[name]
    download(row["url"], DEST / name, row["bytes"])


already present  DinnerPointAve.rad  (776 bytes)
already present  DinnerPointAve.rd3  (8518392 bytes)


## 3. Delete the profiles that are not used

A file is removed when its name is in the release, it is not `DinnerPointAve.rd3` or `DinnerPointAve.rad`, and it is not one of the survey notes. That removes the other nine lines (`.rd3` and `.rad`) and every `.asc` export, including `DinnerPointAve.asc`.

`DinnerPointAve_gprpy_steps.npz` is not in the release, so this step does not delete it.


In [ ]:
removed = []
for path in sorted(DEST.iterdir()):
    if not path.is_file():
        continue
    row = by_name.get(path.name)
    if row is not None and row["action"] == "delete":
        path.unlink()
        removed.append(path.name)

print(f"removed {len(removed)} files")
for name in removed:
    print(" ", name)
print("left in the folder:")
for path in sorted(DEST.iterdir()):
    print(f"  {path.name}  {path.stat().st_size} bytes")


removed 18 files
  GameFarmRd-1.rad
  GameFarmRd-1.rd3
  GameFarmRd-2.rad
  GameFarmRd-2.rd3
  ReedyCreek-1.rad
  ReedyCreek-1.rd3
  ReedyCreek-2.rad
  ReedyCreek-2.rd3
  ScottsLandingRd.rad
  ScottsLandingRd.rd3
  SouthWildlifeDr.rad
  SouthWildlifeDr.rd3
  StaffordAve-eastwest.rad
  StaffordAve-eastwest.rd3
  StaffordAve-westeast.rad
  StaffordAve-westeast.rd3
  WescottAve.rad
  WescottAve.rd3
left in the folder:
  DinnerPointAve.rad  776 bytes
  DinnerPointAve.rd3  8518392 bytes
  DinnerPointAve_gprpy_steps.npz  102264090 bytes
  Raw ground-penetrating radar data, Edwin B. Forsythe National Wildlife Refuge, New Jersey, 2014-15.xml  21777 bytes
  TraceCoordinates.txt  98034 bytes
  markers.txt  5113 bytes
